In [ ]:
Аналіз A/B-тестів

Cookie Cats - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. 
Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань я проаналізую результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. 
Зокрема, хочу зрозуміти, як це вплинуло на утримання (retention) гравців, тобто чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на поведінку гравців.
Особливу увагу приділю на те, чи користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Працюватиму з даними з файлу cookie_cats.csv. Колонки в даних наступні:

userid - унікальний номер, який ідентифікує кожного гравця.
version - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
sum_gamerounds - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
retention_1 - чи через 1 день після встановлення гравець повернувся і почав грати?
retention_7 - чи через 7 днів після встановлення гравець повернувся і почав грати?
Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40

In [5]:
import pandas as pd
import numpy as np
from scipy import stats

In [8]:
data = pd.read_csv("cookie_cats.csv")
data.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [ ]:
Для початку, уявімо, що я тільки планую проведення зазначеного А/B-тесту і хочу зрозуміти, дані про скількох користувачів мені
треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом вважається збільшення утримання на 1% після внесення зміни. 
Обчислю, скільки користувачів сумарно мені треба аби досягнути такого ефекту, якщо продакт менеджер мені повідомив, що базове 
утримання - 19%.

In [9]:
p1 = 0.19 #поточне утримування
p2 = 0.2 #очікуване утримання
alpha = 0.05
power = 0.8

p_bar = (p1 + p2) / 2 #середня пропорція
z_alpha = stats.norm.ppf(1-alpha / 2)
z_beta = stats.norm.ppf(power)

n = ((z_alpha * np.sqrt(2 * p_bar * (1 - p_bar)) + z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2 / (p1 - p2) ** 2)
n_group = int(np.ceil(n))
n_total = 2 * n_group

print(f'Необхідна кількість користувачів для однієї групи: {n_group}')
print(f'Необхідна кількість користувачів загалом: {n_total}')

Необхідна кількість користувачів для однієї групи: 24641
Необхідна кількість користувачів загалом: 49282


In [10]:
retention_groups = data.groupby('version')['retention_7'].mean()
print(retention_groups)

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


In [ ]:
Проаналізувавши середнє значення retention для тестової та контрольної групи , можна помітити, що gate_30 має кращий рівень 
утримання на рівні 19% проти утримання в gate_40 на равні 18,2%. Щоб перевірити, чи ця різниця статистично значуща, а не випадкова,
формулюю гіпотези:

H0: p30 = p40
H1: p30 ≠ p40

In [ ]:
Досліджую наступні питання:
Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?
Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?

In [13]:
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

g = data.groupby('version')['retention_7'].agg(['sum', 'count'])
count = g.loc[['gate_30','gate_40'],'sum'].to_numpy() #кількість успіхів
nobs = g.loc[['gate_30','gate_40'],'count'].to_numpy() #кількість спостережень

stat, pval = proportions_ztest(count, nobs, alternative="two-sided")
(low_30, low_40), (up_30, up_40) = proportion_confint(count, nobs, alpha = 0.05)

print(f'z statistic: {stat:.2f}')
print(f'p-value {pval:.4f}')
print(f'Довірчий інтервал 95% для групи control: [{low_30:.3f}, {up_30:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{low_40:.3f}, {up_40:.3f}]')

z statistic: 3.16
p-value 0.0016
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


In [ ]:
Різниця є статистично значущою, оскільки p(0.0016) < p(0.05) і є достатньо підстав відхилити Н0 гіпотезу.
Інтервали двох груп не перетинаються, тобто немає правдоподібних значень для обох версій одночасно.
Це свідчить про статистично значущу різницю, яка згадується вище. 
З довірою 95% можна стверджувати, що утримання на 7-ий день для групи gate_30 є вищим в порівнянні з gate_40

In [ ]:
Виконую тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7-ий день 
після реєстрації.

In [14]:
crosstab = pd.crosstab(data['version'], data['retention_7'])
crosstab

retention_7,False,True
version,,
gate_30,36198,8502
gate_40,37210,8279


In [15]:
chi2, p, dof, expected = stats.chi2_contingency(crosstab)
alpha = 0.05
chi2_crit = stats.chi2.ppf(1 - alpha, dof)

print (f'χ² = {chi2:.3f}')
print (f'p-value = {p:.5f}')
print (f'Ступені свободи = {dof}')
print (f'Очікувані частоти:\n', expected)
print (f'Критичне значення = {chi2_crit:.3f}')

χ² = 9.959
p-value = 0.00160
Ступені свободи = 1
Очікувані частоти:
 [[36382.90257127  8317.09742873]
 [37025.09742873  8463.90257127]]
Критичне значення = 3.841


In [ ]:
Оскільки значення χ² є більшим за критичне значення (9.959 > 3.841), а значення p < 0,05, то на рівні значущості 5% відхиляю H0.
Це демонструє, що утримання на 7 день залежить від того, на якому рівні розташовані ворота. Утримання в gate_30 (≈ 19,0%) вище, 
ніж у gate_40 (≈ 18,2%), тобто перенесення першої брами з 30-го на 40-й рівень, імовірно, знизило утримання